<style>
body {
    background: linear-gradient(135deg, #f3e7ff 0%, #e3f0ff 100%);
    font-family: 'Segoe UI', Arial, sans-serif;
}
div.custom-section {
    border: 2px solid #c77dff;
    padding: 18px;
    border-radius: 12px;
    background-color: #f8f0ff;
    box-shadow: 0 2px 8px rgba(123,44,191,0.08);
}
h2 {
    color: #7b2cbf;
}
</style>

<div class="custom-section">
<h2>Limpeza e Tratamento dos Dados</h2>
</div>

In [49]:
# Leitura do arquivo CSV
import os


nome_pasta = input("Digite seu nome (pasta): ")
base_path = f"../database/{nome_pasta}/Takeout/YouTube e YouTube Music"
os.listdir(base_path)

['canais',
 'chats ao vivo',
 'comentários',
 'histórico',
 'inscrições',
 'playlists']

In [50]:
from pathlib import Path
from bs4 import BeautifulSoup
import pandas as pd

def ler_e_exibir_csv(caminho_arquivo):
   
    return pd.read_csv(caminho_arquivo)


def extrair_historico_youtube(caminho_arquivo):
    with open(caminho_arquivo, "r", encoding="utf-8") as f:
        soup = BeautifulSoup(f, "html.parser")

    dados = []

    blocos = soup.find_all("div", class_="content-cell")

    for bloco in blocos:
        a_tag = bloco.find("a")
        
        if a_tag:
            titulo = a_tag.get_text(strip=True)
            link = a_tag["href"]
            
            texto_completo = bloco.get_text(separator="\n").split("\n")
            
            # Tipo de ação
            if "Watched" in texto_completo[0]:
                tipo = "watch"
            elif "Searched for" in texto_completo[0]:
                tipo = "search"
            else:
                continue
            
            # Data
            data = [t for t in texto_completo if "BRT" in t]
            data = data[0] if data else None
            
            dados.append([tipo, titulo, data, link])

    df = pd.DataFrame(dados, columns=["tipo", "titulo", "data", "link"])
    
    # 🔹 Limpeza da data
    df["data"] = (
        df["data"]
        .str.replace(" BRT", "", regex=False)  # remove timezone textual
    )

    # 🔹 Conversão segura e padronizada
    df["data"] = pd.to_datetime(
        df["data"],
        format="%b %d, %Y, %I:%M:%S %p",
        errors="coerce"
    )

    return df


In [51]:
pesquisa = extrair_historico_youtube(base_path+"/histórico/histórico de pesquisa.html")
print(pesquisa)

        tipo                                             titulo data  \
0      watch     Mycon | Como todo consórcio deveria ser (16x9)  NaT   
1     search                                       cristo jesus  NaT   
2      watch     Jornada Python - ADV LPY25 04h sem data Newton  NaT   
3      watch  CAGE0230.Google.PWA.Contentformance_3mil.Calcu...  NaT   
4      watch                        Linha Can-Am 2026: Defender  NaT   
...      ...                                                ...  ...   
1062  search                               colocacao pronominal  NaT   
1063  search                  santo agostinho e tomás de aquino  NaT   
1064  search                                       etica crista  NaT   
1065  search                                     genetica 9 ano  NaT   
1066  search                                   string invertida  NaT   

                                                   link  
0           https://www.youtube.com/watch?v=EPuSc-G3v9A  
1     https://www.y

In [52]:
visual = extrair_historico_youtube(base_path+"/histórico/histórico-de-visualização.html")
print(visual)

       tipo                                             titulo data  \
0     watch  Deseable - Brunão Morada + Marcos Brunet // So...  NaT   
1     watch     Mycon | Como todo consórcio deveria ser (16x9)  NaT   
2     watch  Cristo Jesus (Ao Vivo) - fhop music & Paulo e ...  NaT   
3     watch     Jornada Python - ADV LPY25 04h sem data Newton  NaT   
4     watch  CAGE0230.Google.PWA.Contentformance_3mil.Calcu...  NaT   
...     ...                                                ...  ...   
1632  watch                           Ética Cristã - FILOSOFIA  NaT   
1633  watch  GENÉTICA: LEIS DE MENDEL, GENES, DNA E CROMOSS...  NaT   
1634  watch                 Como inverter uma string em Python  NaT   
1635  watch               Curso Python #20 - Funções (Parte 1)  NaT   
1636  watch  Como instalar e testar o Python 3.9 no Windows 10  NaT   

                                             link  
0     https://www.youtube.com/watch?v=9jPJfWqT-So  
1     https://www.youtube.com/watch?v=EPuSc

In [53]:
inscricoes = ler_e_exibir_csv(base_path+"/inscrições/inscrições.csv")
print(inscricoes)

                ID do canal  \
0  UCC_4Qpl1-Kysd-9hRTsoX8w   
1  UCGnsRwFS8Xm13lMAeouzT9g   
2  UCP_lo1MFyx5IXDeD9s_6nUw   
3  UCbwRqQqZh1ycZ_cZw3rjOLA   
4  UCePyJZBjplMkrMp0kW-HxFg   
5  UChzps1qR8zGLHCDQ5E51GOA   
6  UCwSVZMy2NHxdmVt2-7EeNOA   

                                        URL do canal   Título do canal  
0  http://www.youtube.com/channel/UCC_4Qpl1-Kysd-...      WoMakersCode  
1  http://www.youtube.com/channel/UCGnsRwFS8Xm13l...    Allie Sherlock  
2  http://www.youtube.com/channel/UCP_lo1MFyx5IXD...   Meta Developers  
3  http://www.youtube.com/channel/UCbwRqQqZh1ycZ_...      Calebe Cravo  
4  http://www.youtube.com/channel/UCePyJZBjplMkrM...  The Dream School  
5  http://www.youtube.com/channel/UChzps1qR8zGLHC...          Repo Dev  
6  http://www.youtube.com/channel/UCwSVZMy2NHxdmV...    Lorena Vieira   


In [72]:
print("Data mínima:", visual['data'].min())
print("Data máxima:", visual['data'].max())
print("Limite:", limite)

Data mínima: NaT
Data máxima: NaT
Limite: 2026-01-25 23:40:34.486372


In [65]:
visual['data'].isna().sum()

np.int64(0)

In [69]:
visual['data']=pd.to_datetime(visual['data'],dayfirst=True, errors='coerce')
pesquisa['data']=pd.to_datetime(pesquisa['data'],dayfirst=True, errors='coerce')

limite=pd.Timestamp.today()-pd.Timedelta(days=30)

visual=visual[visual['data']>=limite]
pesquisa=pesquisa[pesquisa['data']>=limite]

In [71]:
print(visual)

Empty DataFrame
Columns: [tipo, titulo, data, link, proximo_video]
Index: []


In [55]:
visual['titulo'].isna().sum()
pesquisa['titulo'].isna().sum()
inscricoes['Título do canal'].isna().sum()

# Nenhum dado nulo foi encontrado

np.int64(0)

In [56]:
visual=visual.drop_duplicates()
pesquisa=pesquisa.drop_duplicates()

In [57]:
import re

def limpar_texto(texto):
    texto=texto.lower()
    texto=re.sub(r'[^\w\s]','',texto)
    return texto

visual['titulo']=visual['titulo'].apply(limpar_texto)
pesquisa['titulo']=pesquisa['titulo'].apply(limpar_texto)

In [ ]:
print(visual)

,tipo,titulo,data,link,proximo_video


<style>
body {
    background: linear-gradient(135deg, #f3e7ff 0%, #e3f0ff 100%);
    font-family: 'Segoe UI', Arial, sans-serif;
}
div.custom-section {
    border: 2px solid #c77dff;
    padding: 18px;
    border-radius: 12px;
    background-color: #f8f0ff;
    box-shadow: 0 2px 8px rgba(123,44,191,0.08);
}
h2 {
    color: #7b2cbf;
}
</style>

<div class="custom-section">

<h2>Séries Temporais</h2>
</div>

In [ ]:
visual=visual.sort_values('data')
visual['proximo_video']=visual['titulo'].shift(-1)

In [ ]:
print(visual)

,tipo,titulo,data,link,proximo_video
